# 02 · Silver layer – Nordvia Retail (SAP FI)

**Goal.** Turn the 11 raw CSV tables (bronze) into typed, consistently formatted tables (silver) in which every finding of the data quality log is visible as a column. Silver is the input of the gold layer, where the tables are reshaped for the ontology.

**What silver does**
1. Applies the README conventions: dates, amounts, flags, language and country codes.
2. Marks every row affected by an entry of `data_quality/data_quality_log.csv` with a flag column.
3. Checks that each flag marks exactly the rows the log reports.

**What silver does not do.** It deletes no row and corrects no business value. The decisions (what to exclude, what to merge, how to convert to euros) are taken in gold.

**Reading the columns.** Columns that come from SAP keep their technical name in upper case (`BUKRS`, `BUDAT`). Columns added here are in lower case: `is_…` describes what the row is, `dq_…` marks a finding of the log.

## 0 · Environment

In [1]:

import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
import os, sys
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

print("Python", sys.version.split()[0], "| pandas", pd.__version__)

DATA_PATH = "data"            
SILVER_PATH = "silver"
LOG_PATH = "data_quality/data_quality_log.csv"
YEAR_START, KEY_DATE = "20250101", "20251231"

TABLES = ["T001", "SKA1", "SKAT", "SKB1", "CSKS", "KNA1", "LFA1", "T003", "TBSL", "BKPF", "BSEG"]

Python 3.12.7 | pandas 3.1.1


## 1 · Bronze
Read exactly as in notebook 01: everything as text, empty strings kept as they are. Bronze is never modified; every silver table starts from a copy.

In [3]:
bronze = {t: pd.read_csv(f"{DATA_PATH}/{t}.csv", dtype=str, keep_default_na=False, encoding="utf-8") for t in TABLES}
pd.DataFrame({"rows": {t: len(d) for t, d in bronze.items()}, "columns": {t: d.shape[1] for t, d in bronze.items()}})

,rows,columns
T001,3,8
SKA1,27,7
SKAT,55,5
SKB1,80,7
CSKS,25,9
KNA1,156,10
LFA1,72,10
T003,7,3
TBSL,8,4
BKPF,2353,15


## 2 · Formatting rules
One function per README convention, so each rule is written once.

| Rule | Source | Function |
|---|---|---|
| Identifiers keep their leading zeros: they stay text | README | *(nothing to do)* |
| Dates `YYYYMMDD` become ISO `YYYY-MM-DD`; `99991231` (no end date) is kept as `9999-12-31` | README | `iso_date` |
| Amounts and rates become numbers; amounts stay unsigned | README | `number` |
| `X` flags become true / false | README | `is_x` |
| Language keys `E` / `S` become the tags `en` / `es` | README | `LANGUAGE` |
| An empty string means "no value" | CSV | `finish` |

Dates are kept as ISO **text** rather than as a datetime type: `9999-12-31` does not fit every datetime type, and ISO text sorts and compares correctly.

In [4]:
def iso_date(s):
    """20250630 -> 2025-06-30; an empty value stays empty."""
    return (s.str[:4] + "-" + s.str[4:6] + "-" + s.str[6:]).where(s != "", "")

def number(s):
    """Text -> number; an empty value becomes missing."""
    return pd.to_numeric(s.where(s != ""))

def is_x(s):
    """SAP flag: 'X' -> True, anything else -> False."""
    return s == "X"

LANGUAGE = {"E": "en", "S": "es"}

silver = {}

def finish(name, d):
    """Store a silver table: empty strings become missing values."""
    silver[name] = d.replace("", pd.NA)
    return silver[name]

K = ["BUKRS", "BELNR", "GJAHR"]           

## 3 · Master data and configuration
### 3.1 T001 · Company codes
`local_currency_is_euro` shows which company codes need a conversion to report in euros (log: no exchange-rate table).

In [5]:
d = bronze["T001"].copy()
d["language_tag"] = d["SPRAS"].map(LANGUAGE)
d["local_currency_is_euro"] = d["WAERS"] == "EUR"
finish("T001", d)

,BUKRS,BUTXT,ORT01,LAND1,WAERS,SPRAS,KTOPL,PERIV,language_tag,local_currency_is_euro
0,1000,Nordvia Retail Iberia S.L.,Barcelona,ES,EUR,S,NVCA,K4,es,True
1,2000,Nordvia Retail US Inc.,Chicago,US,USD,E,NVCA,K4,en,False
2,3000,Nordvia Retail UK Ltd.,Manchester,GB,GBP,E,NVCA,K4,en,False


### 3.2 SKA1 and SKAT · G/L accounts and their texts
- `SKA1`: the two type flags become booleans; `dq_missing_spanish_text` marks the account that only has an English text.
- `SKAT`: `dq_no_account_definition` marks the texts of an account that is not in the chart.

In [6]:
skat0 = bronze["SKAT"]
languages = skat0.groupby("SAKNR")["SPRAS"].agg(set)

d = bronze["SKA1"].copy()
d["ERDAT"] = iso_date(d["ERDAT"])
d["is_balance_sheet"] = is_x(d["XBILK"])
d["is_profit_and_loss"] = is_x(d["GVTYP"])
d["dq_missing_spanish_text"] = ~d["SAKNR"].map(lambda a: "S" in languages.get(a, set()))
finish("SKA1", d)

d = skat0.copy()
d["language_tag"] = d["SPRAS"].map(LANGUAGE)
d["dq_no_account_definition"] = ~d["SAKNR"].isin(bronze["SKA1"]["SAKNR"])
finish("SKAT", d)
silver["SKA1"].head()

,KTOPL,SAKNR,KTOKS,XBILK,GVTYP,ERDAT,ERNAM,is_balance_sheet,is_profit_and_loss,dq_missing_spanish_text
0,NVCA,0000100000,SAKO,X,NaN,2019-03-15,MIGRATION,True,False,False
1,NVCA,0000113100,SAKO,X,NaN,2019-03-15,MIGRATION,True,False,False
2,NVCA,0000140000,SAKO,X,NaN,2019-03-15,MIGRATION,True,False,False
3,NVCA,0000154000,SAKO,X,NaN,2019-03-15,MIGRATION,True,False,False
4,NVCA,0000160000,SAKO,X,NaN,2019-03-15,MIGRATION,True,False,False


### 3.3 SKB1 · G/L accounts per company code
`cost_center_required` is the README rule `FSTAG = G004`, used later to find postings without a cost center.

In [7]:
d = bronze["SKB1"].copy()
d["is_open_item_managed"] = is_x(d["XOPVW"])
d["has_line_item_display"] = is_x(d["XKRES"])
d["cost_center_required"] = d["FSTAG"] == "G004"
finish("SKB1", d).head()

,BUKRS,SAKNR,WAERS,MITKZ,XOPVW,FSTAG,XKRES,is_open_item_managed,has_line_item_display,cost_center_required
0,1000,0000100000,EUR,NaN,NaN,G001,X,False,True,False
1,2000,0000100000,USD,NaN,NaN,G001,X,False,True,False
2,3000,0000100000,GBP,NaN,NaN,G001,X,False,True,False
3,1000,0000113100,EUR,NaN,NaN,G001,X,False,True,False
4,2000,0000113100,USD,NaN,NaN,G001,X,False,True,False


### 3.4 CSKS · Cost centers
`99991231` is kept as the date `9999-12-31`, so that validity is a plain comparison (posting date ≤ valid to); `is_open_ended` says explicitly that it is not a real end date.

In [8]:
d = bronze["CSKS"].copy()
d["is_open_ended"] = d["DATBI"] == "99991231"
d["DATAB"], d["DATBI"] = iso_date(d["DATAB"]), iso_date(d["DATBI"])
finish("CSKS", d)
silver["CSKS"][~silver["CSKS"]["is_open_ended"]]

,KOKRS,KOSTL,DATBI,DATAB,BUKRS,KOSAR,VERAK,KTEXT,LTEXT,is_open_ended
7,NV00,0000011080,2025-06-30,2019-01-01,1000,W,S. Gil,Store Valencia Centr,Store Valencia Centro,False


### 3.5 KNA1 · Customers
- The country is put in upper case; the original value stays in `LAND1_raw` and the rows are marked.
- `is_one_time` and `is_flagged_for_deletion` expose two conditions of the log.
- **One row is added.** Customer `0000100999` is posted on three open invoices but has no master record. It is created here with its number only and `is_inferred = True`, so its receivables are not lost in the joins. It is the only row silver adds to any table.

In [9]:
d = bronze["KNA1"].copy()
d["LAND1_raw"] = d["LAND1"]
d["LAND1"] = d["LAND1"].str.upper()
d["dq_country_lowercase"] = d["LAND1_raw"] != d["LAND1"]
d["ERDAT"] = iso_date(d["ERDAT"])
d["is_flagged_for_deletion"] = is_x(d["LOEVM"])
d["is_one_time"] = d["KTOKD"] == "CPD"
d["is_inferred"] = False

used = bronze["BSEG"].loc[bronze["BSEG"]["KUNNR"] != "", "KUNNR"].unique()
missing = sorted(set(used) - set(d["KUNNR"]))
inferred = pd.DataFrame({"KUNNR": missing, "is_inferred": True, "dq_country_lowercase": False,
                         "is_flagged_for_deletion": False, "is_one_time": False})
d = pd.concat([d, inferred], ignore_index=True).fillna({c: "" for c in bronze["KNA1"].columns} | {"LAND1_raw": ""})
finish("KNA1", d)
silver["KNA1"][silver["KNA1"][["dq_country_lowercase", "is_flagged_for_deletion", "is_one_time", "is_inferred"]].any(axis=1)]

,KUNNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKD,ERDAT,LOEVM,ZTERM,LAND1_raw,dq_country_lowercase,is_flagged_for_deletion,is_one_time,is_inferred
16,0000100017,Coral Retail S.A.,ES,Madrid,28001,A16525001,KUNA,2018-05-14,NaN,NT15,es,True,False,False,False
22,0000100023,Harbor Comercial S.A.,ES,Bilbao,48001,A88133160,KUNA,2024-05-24,X,0001,ES,False,True,False,False
40,0000100041,Orion Distribuciones S.A.,ES,Barcelona,08001,B83650017,KUNA,2024-04-10,NaN,NT15,es,True,False,False,False
153,0000100154,One-time customer,ES,NaN,NaN,NaN,CPD,2019-01-01,NaN,0001,ES,False,False,True,False
154,0000100155,One-time customer,US,NaN,NaN,NaN,CPD,2019-01-01,NaN,0001,US,False,False,True,False
155,0000100156,One-time customer,GB,NaN,NaN,NaN,CPD,2019-01-01,NaN,0001,GB,False,False,True,False
156,0000100999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True


### 3.6 LFA1 · Vendors and employees
- `is_employee` separates the employees paid through the vendor ledger from real vendors.
- `dq_duplicate_tax_id` marks the two records of the same company. `canonical_lifnr` gives, for every record, the lowest vendor number with the same Tax ID (its own number when there is no duplicate). Silver only records the link; whether the two are merged is decided in gold.

In [10]:
d = bronze["LFA1"].copy()
d["ERDAT"] = iso_date(d["ERDAT"])
d["is_blocked"] = is_x(d["SPERR"])
d["is_employee"] = d["KTOKK"] == "EMPL"
has_tax = d["STCD1"] != ""
d["dq_duplicate_tax_id"] = has_tax & d.duplicated("STCD1", keep=False)
d["canonical_lifnr"] = d["LIFNR"].where(~has_tax, d.groupby("STCD1")["LIFNR"].transform("min"))
finish("LFA1", d)
silver["LFA1"][silver["LFA1"]["is_employee"] | silver["LFA1"]["dq_duplicate_tax_id"]]

,LIFNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKK,ERDAT,SPERR,ZTERM,is_blocked,is_employee,dq_duplicate_tax_id,canonical_lifnr
3,0000200004,Harbor Systems S.L.U.,ES,Bilbao,48001,A32661340,LIEF,2019-08-19,NaN,NT45,False,False,True,0000200004
65,0000200066,Laura Martínez Soler,ES,Zaragoza,50001,NaN,EMPL,2022-01-10,NaN,0001,False,True,False,0000200066
66,0000200067,Jordi Casas Pons,ES,Bilbao,48001,NaN,EMPL,2022-01-10,NaN,0001,False,True,False,0000200067
67,0000200068,Michael Turner,US,Austin,73301,NaN,EMPL,2022-01-10,NaN,0001,False,True,False,0000200068
68,0000200069,Ashley Reed,US,Chicago,60601,NaN,EMPL,2022-01-10,NaN,0001,False,True,False,0000200069
69,0000200070,Oliver Bennett,GB,Bristol,BS1 4DJ,NaN,EMPL,2022-01-10,NaN,0001,False,True,False,0000200070
70,0000200071,Sophie Hall,GB,Manchester,M1 1AA,NaN,EMPL,2022-01-10,NaN,0001,False,True,False,0000200071
71,0000200072,HARBOR SYSTEMS SLU,ES,Bilbao,48001,A32661340,LIEF,2025-03-12,NaN,NT45,False,False,True,0000200004


### 3.7 T003 and TBSL · Document types and posting keys
Small code lists with nothing to convert; they are copied so that silver is complete.

In [11]:
finish("T003", bronze["T003"].copy())
finish("TBSL", bronze["TBSL"].copy())
silver["T003"]

,BLART,LTEXT,NUMKR
0,DR,Customer invoice,18
1,DG,Customer credit memo,16
2,DZ,Customer payment,14
3,KR,Vendor invoice,19
4,KZ,Vendor payment,15
5,SA,G/L account document,01
6,AB,Accounting document (reversals),02


## 4 · Documents
### 4.1 BKPF · Document headers
Typed: the three dates, fiscal year, period and exchange rate. Flags:

| Column | Meaning | Log |
|---|---|---|
| `is_reversal_document` | type `AB`, posted to cancel another document | DQ-14 |
| `is_reversed` | original that was cancelled by a reversal | DQ-14 |
| `reversed_in_later_period` | the reversal was posted in a later period than this original | DQ-14 |
| `is_opening_balance` | balances brought forward by the migration user | DQ-15 |
| `is_foreign_currency` | document currency differs from the local currency of the company | DQ-21 |
| `dq_no_line_items` | header without lines | DQ-04 |
| `dq_probable_duplicate_invoice` | same vendor and invoice number as another vendor invoice | DQ-11 |
| `dq_reference_not_unique` | customer invoice whose reference is used by another invoice of the company | DQ-12 |
| `dq_document_date_prior_year` | document date before the fiscal year | DQ-13 |
| `dq_entered_after_key_date` | keyed in after the key date | DQ-13 |

In [12]:
b0, s0 = bronze["BKPF"], bronze["BSEG"]
d = b0.copy()

d["is_reversal_document"] = d["BLART"] == "AB"
d["is_reversed"] = (d["STBLG"] != "") & ~d["is_reversal_document"]
partner = d.merge(b0[K + ["MONAT"]], how="left", left_on=["BUKRS", "STBLG", "STJAH"], right_on=K, suffixes=("", "_partner"))
d["reversed_in_later_period"] = d["is_reversed"] & (partner["MONAT_partner"].fillna(d["MONAT"]) > d["MONAT"]).values

d["is_opening_balance"] = d["USNAM"] == "MIGRATION"
d["is_foreign_currency"] = d["WAERS"] != d["BUKRS"].map(bronze["T001"].set_index("BUKRS")["WAERS"])
d["dq_no_line_items"] = ~d.set_index(K).index.isin(s0.set_index(K).index)

# vendor invoices entered twice: same vendor and same invoice number
vendor = s0[s0["KOART"] == "K"].drop_duplicates(K)[K + ["LIFNR"]]
kr = d[(d["BLART"] == "KR") & (d["XBLNR"] != "")].merge(vendor, on=K)
dup = kr[kr.duplicated(["LIFNR", "XBLNR"], keep=False)]
d["dq_probable_duplicate_invoice"] = d.set_index(K).index.isin(dup.set_index(K).index)

# customer invoices sharing a reference inside one company code
dr = d[(d["BLART"] == "DR") & (d["XBLNR"] != "")]
d["dq_reference_not_unique"] = d.index.isin(dr[dr.duplicated(["BUKRS", "XBLNR"], keep=False)].index)

d["dq_document_date_prior_year"] = d["BLDAT"] < YEAR_START
d["dq_entered_after_key_date"] = d["CPUDT"] > KEY_DATE

for c in ["BLDAT", "BUDAT", "CPUDT"]:
    d[c] = iso_date(d[c])
d["GJAHR_n"], d["MONAT_n"], d["KURSF"] = d["GJAHR"].astype(int), d["MONAT"].astype(int), number(d["KURSF"])
finish("BKPF", d)
flags_bkpf = [c for c in d.columns if c.startswith(("is_", "dq_", "reversed_"))]
silver["BKPF"][flags_bkpf].sum().to_frame("documents")

,documents
is_reversal_document,7
is_reversed,7
reversed_in_later_period,2
is_opening_balance,3
is_foreign_currency,76
dq_no_line_items,1
dq_probable_duplicate_invoice,2
dq_reference_not_unique,2
dq_document_date_prior_year,8
dq_entered_after_key_date,13


`GJAHR` and `MONAT` keep their text form because they are part of keys; `GJAHR_n` and `MONAT_n` are the same values as numbers.

### 4.2 BSEG · Line items
Typed: the two amounts, the line number and the two dates. Amounts stay unsigned, as in the source; the sign is still in `SHKZG`. Flags:

| Column | Meaning | Log |
|---|---|---|
| `is_open_item` | customer or vendor line without a clearing document | reviewed |
| `is_open_credit_memo` | open line of a customer credit memo | DQ-20 |
| `is_fx_difference` | exchange-rate difference: no document amount, only a local amount | DQ-16 |
| `dq_local_amount_not_at_document_rate` | local amount is not document amount × rate (payment settled at another rate) | DQ-16 |
| `dq_customer_not_in_master` | customer number without a record in KNA1 | DQ-01 |
| `dq_account_not_in_company` | account not set up in SKB1 for the company code | DQ-02 |
| `dq_missing_cost_center` | account requires a cost center and the line has none | DQ-17 |
| `dq_cost_center_expired` | posted outside the validity of the cost center | DQ-18 |
| `dq_terms_differ_from_master` | payment terms differ from the customer master | DQ-19 |

In [13]:
d = s0.merge(b0[K + ["BLART", "BUDAT", "KURSF"]], on=K, how="left")        
wrbtr, dmbtr = number(d["WRBTR"]), number(d["DMBTR"])

d["is_open_item"] = d["KOART"].isin(["D", "K"]) & (d["AUGBL"] == "")
d["is_open_credit_memo"] = d["is_open_item"] & (d["BLART"] == "DG")
d["is_fx_difference"] = wrbtr == 0
d["dq_local_amount_not_at_document_rate"] = (wrbtr != 0) & ((dmbtr - wrbtr * number(d["KURSF"])).abs() > 0.02)

d["dq_customer_not_in_master"] = (d["KUNNR"] != "") & ~d["KUNNR"].isin(bronze["KNA1"]["KUNNR"])

skb1 = bronze["SKB1"].set_index(["BUKRS", "SAKNR"])
account = pd.MultiIndex.from_frame(d[["BUKRS", "HKONT"]])
d["dq_account_not_in_company"] = ~account.isin(skb1.index)
required = skb1.index[skb1["FSTAG"] == "G004"]
d["dq_missing_cost_center"] = account.isin(required) & (d["KOSTL"] == "")

validity = bronze["CSKS"].set_index("KOSTL")
valid_from, valid_to = d["KOSTL"].map(validity["DATAB"]), d["KOSTL"].map(validity["DATBI"])
d["dq_cost_center_expired"] = (d["KOSTL"] != "") & ((d["BUDAT"] < valid_from) | (d["BUDAT"] > valid_to))

master_terms = d["KUNNR"].map(bronze["KNA1"].set_index("KUNNR")["ZTERM"])
d["dq_terms_differ_from_master"] = (d["KUNNR"] != "") & (d["ZTERM"] != "") & master_terms.notna() & (d["ZTERM"] != master_terms)

d = d.drop(columns=["BLART", "BUDAT", "KURSF"])                              
d["WRBTR"], d["DMBTR"], d["BUZEI_n"] = wrbtr, dmbtr, d["BUZEI"].astype(int)
d["ZFBDT"], d["AUGDT"] = iso_date(d["ZFBDT"]), iso_date(d["AUGDT"])
finish("BSEG", d)
flags_bseg = [c for c in d.columns if c.startswith(("is_", "dq_"))]
silver["BSEG"][flags_bseg].sum().to_frame("lines")

,lines
is_open_item,167
is_open_credit_memo,22
is_fx_difference,36
dq_local_amount_not_at_document_rate,36
dq_customer_not_in_master,3
dq_account_not_in_company,2
dq_missing_cost_center,5
dq_cost_center_expired,4
dq_terms_differ_from_master,11


## 5 · Checks
### 5.1 Nothing was lost
Every silver table has the rows of its bronze table. The only difference allowed is the customer added in `KNA1`. The source columns are all still there, and the amounts add up to the same totals.

In [14]:
rows = pd.DataFrame({"bronze": {t: len(bronze[t]) for t in TABLES}, "silver": {t: len(silver[t]) for t in TABLES}})
rows["difference"] = rows["silver"] - rows["bronze"]
assert rows.drop("KNA1")["difference"].eq(0).all() and rows.loc["KNA1", "difference"] == silver["KNA1"]["is_inferred"].sum()
assert all(set(bronze[t].columns) <= set(silver[t].columns) for t in TABLES)
for c in ["WRBTR", "DMBTR"]:
    assert round(silver["BSEG"][c].sum(), 2) == round(pd.to_numeric(bronze["BSEG"][c]).sum(), 2)
rows

,bronze,silver,difference
T001,3,3,0
SKA1,27,27,0
SKAT,55,55,0
SKB1,80,80,0
CSKS,25,25,0
KNA1,156,157,1
LFA1,72,72,0
T003,7,7,0
TBSL,8,8,0
BKPF,2353,2353,0


### 5.2 The flags match the log
For each entry of the data quality log, the number of rows marked in silver is compared with the number of affected rows the log reports. Any difference stops the notebook.

In [15]:
log = pd.read_csv(LOG_PATH, dtype=str).set_index("id")
S = silver
checks = [
    ("DQ-01", "BSEG", "dq_customer_not_in_master",        S["BSEG"]["dq_customer_not_in_master"]),
    ("DQ-02", "BSEG", "dq_account_not_in_company",        S["BSEG"]["dq_account_not_in_company"]),
    ("DQ-03", "SKAT", "dq_no_account_definition",         S["SKAT"]["dq_no_account_definition"]),
    ("DQ-04", "BKPF", "dq_no_line_items",                 S["BKPF"]["dq_no_line_items"]),
    ("DQ-05", "SKA1", "dq_missing_spanish_text",          S["SKA1"]["dq_missing_spanish_text"]),
    ("DQ-06", "KNA1", "dq_country_lowercase",             S["KNA1"]["dq_country_lowercase"]),
    ("DQ-07", "KNA1", "is_one_time",                      S["KNA1"]["is_one_time"]),
    ("DQ-08", "KNA1", "is_flagged_for_deletion",          S["KNA1"]["is_flagged_for_deletion"]),
    ("DQ-09", "LFA1", "dq_duplicate_tax_id",              S["LFA1"]["dq_duplicate_tax_id"]),
    ("DQ-10", "LFA1", "is_employee",                      S["LFA1"]["is_employee"]),
    ("DQ-11", "BKPF", "dq_probable_duplicate_invoice",    S["BKPF"]["dq_probable_duplicate_invoice"]),
    ("DQ-12", "BKPF", "dq_reference_not_unique",          S["BKPF"]["dq_reference_not_unique"]),
    ("DQ-13", "BKPF", "dq_document_date_prior_year or dq_entered_after_key_date",
                                                          S["BKPF"]["dq_document_date_prior_year"] | S["BKPF"]["dq_entered_after_key_date"]),
    ("DQ-14", "BKPF", "is_reversal_document or is_reversed", S["BKPF"]["is_reversal_document"] | S["BKPF"]["is_reversed"]),
    ("DQ-15", "BKPF", "is_opening_balance",               S["BKPF"]["is_opening_balance"]),
    ("DQ-16", "BSEG", "is_fx_difference",                 S["BSEG"]["is_fx_difference"]),
    ("DQ-17", "BSEG", "dq_missing_cost_center",           S["BSEG"]["dq_missing_cost_center"]),
    ("DQ-18", "BSEG", "dq_cost_center_expired",           S["BSEG"]["dq_cost_center_expired"]),
    ("DQ-19", "BSEG", "dq_terms_differ_from_master",      S["BSEG"]["dq_terms_differ_from_master"]),
    ("DQ-20", "BSEG", "is_open_credit_memo",              S["BSEG"]["is_open_credit_memo"]),
    ("DQ-21", "T001", "not local_currency_is_euro",       ~S["T001"]["local_currency_is_euro"]),
]
recon = pd.DataFrame([{"id": i, "silver_table": t, "flag": f, "rows_flagged": int(s.sum()),
                       "rows_in_log": int(log.loc[i, "affected_rows"]), "finding": log.loc[i, "finding"][:70] + "…"}
                      for i, t, f, s in checks])
recon["match"] = recon["rows_flagged"] == recon["rows_in_log"]
assert set(recon["id"]) == set(log.index), "a log entry has no flag in silver"
assert recon["match"].all(), recon[~recon["match"]]
recon

,id,silver_table,flag,rows_flagged,rows_in_log,finding,match
0,DQ-01,BSEG,dq_customer_not_in_master,3,3,Customer 0000100999 is used on 3 lines but doe...,True
1,DQ-02,BSEG,dq_account_not_in_company,2,2,Account 0000800100 (Revenue - services) is pos...,True
2,DQ-03,SKAT,dq_no_account_definition,2,2,Account 0000455000 has texts in English and Sp...,True
3,DQ-04,BKPF,dq_no_line_items,1,1,Document 0100000050 ('Accrual Q4 - pending') h...,True
4,DQ-05,SKA1,dq_missing_spanish_text,1,1,Account 0000499000 (Miscellaneous expense) onl...,True
5,DQ-06,KNA1,dq_country_lowercase,2,2,2 customers have the country in lowercase ('es...,True
6,DQ-07,KNA1,is_one_time,3,3,"3 customers are one-time accounts (CPD, name '...",True
7,DQ-08,KNA1,is_flagged_for_deletion,1,1,Customer 0000100023 (Harbor Comercial S.A.) is...,True
8,DQ-09,LFA1,dq_duplicate_tax_id,2,2,Vendors 0000200004 and 0000200072 ('Harbor Sys...,True
9,DQ-10,LFA1,is_employee,6,6,6 vendor records are employees (EMPL; 6 withou...,True


All the entries of the log have a flag in silver and the counts agree. Three notes on how to read them:
- **DQ-16** also has `dq_local_amount_not_at_document_rate`, which marks the payment lines whose local amount was settled at another rate; each of them shares a document with one `is_fx_difference` line.
- **DQ-09** is completed by `canonical_lifnr`, and **DQ-14** by `reversed_in_later_period`.
- **DQ-21** has no row to mark: it is a missing table. The flag only shows which company codes need a conversion; the monthly rates are built in gold.

## 6 · Save
One CSV per table in `silver/`, UTF-8, with the same file names as bronze. Missing values are written as empty fields and booleans as `True` / `False`.

In [16]:
os.makedirs(SILVER_PATH, exist_ok=True)
for t in TABLES:
    silver[t].to_csv(f"{SILVER_PATH}/{t}.csv", index=False, encoding="utf-8")
added = {t: [c for c in silver[t].columns if c not in bronze[t].columns] for t in TABLES}
pd.DataFrame({"rows": {t: len(silver[t]) for t in TABLES}, "columns_added": {t: ", ".join(added[t]) for t in TABLES}})

,rows,columns_added
T001,3,"language_tag, local_currency_is_euro"
SKA1,27,"is_balance_sheet, is_profit_and_loss, dq_missi..."
SKAT,55,"language_tag, dq_no_account_definition"
SKB1,80,"is_open_item_managed, has_line_item_display, c..."
CSKS,25,is_open_ended
KNA1,157,"LAND1_raw, dq_country_lowercase, is_flagged_fo..."
LFA1,72,"is_blocked, is_employee, dq_duplicate_tax_id, ..."
T003,7,
TBSL,8,
BKPF,2353,"is_reversal_document, is_reversed, reversed_in..."
